<a href="https://colab.research.google.com/github/nuuuuurinn/Assessing-Racial-Disparities-in-Maternity-Care-with-Machine-Learning/blob/main/Data_Modification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import zipfile
from pathlib import Path
import shutil # Added for forcefully removing directories

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import (
    f1_score,
    recall_score,
    precision_score,
    average_precision_score,
    classification_report,
    confusion_matrix
)
from sklearn.linear_model import LogisticRegression

# =========================
# 1) MOUNT GOOGLE DRIVE
# =========================
from google.colab import drive

mount_point = '/content/drive'

# Forcefully remove the mount point directory if it exists to ensure it's clean
if os.path.exists(mount_point):
    print(f"Forcefully clearing and recreating mount point directory: {mount_point}")
    try:
        shutil.rmtree(mount_point)
    except OSError as e:
        print(f"Error removing {mount_point}: {e}. This might happen if it's currently in use or due to permissions.")
        # If removal fails, try to unmount if it's recognized as a mount point
        if os.path.ismount(mount_point):
            try:
                drive.flush_and_unmount()
                print('Google Drive unmounted successfully before remount attempt.')
            except ValueError:
                print('Google Drive was not mounted or could not be unmounted before remount attempt.')
else:
    print(f"Mount point directory '{mount_point}' does not exist, creating it.")

# Ensure the mount point directory exists before mounting
os.makedirs(mount_point, exist_ok=True)

# Now mount Google Drive, forcing a remount
print("Attempting to mount Google Drive...")
drive.mount(mount_point, force_remount=True)

# =========================
# 2) SET ZIP FILE PATH
# =========================
# CHANGE THIS if your zip is inside a folder in Google Drive.
# Example:
# ZIP_FILE = "/content/drive/MyDrive/Data/Nat2024us.zip"
ZIP_FILE = "/content/drive/MyDrive/Nat2024us.zip"

# Where extracted files and outputs will go
WORK_DIR = Path("/content/nat2024_project")
EXTRACT_DIR = WORK_DIR / "extracted"
OUTPUT_DIR = WORK_DIR / "outputs"

WORK_DIR.mkdir(parents=True, exist_ok=True)
EXTRACT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# =========================
# 3) CHECK ZIP FILE EXISTS
# =========================
if not Path(ZIP_FILE).exists():
    raise FileNotFoundError(
        f"Could not find zip file at:\n{ZIP_FILE}\n\n"
        "Fix the ZIP_FILE path so it matches the location of Nat2024us.zip in your Google Drive."
    )

print("✅ Zip file found:", ZIP_FILE)

# =========================
# 4) UNZIP USING SYSTEM TOOL (FIXED)
# =========================
import subprocess

print("Unzipping with system tool...")

result = subprocess.run(
    ["unzip", "-o", ZIP_FILE, "-d", str(EXTRACT_DIR)],
    capture_output=True,
    text=True
)

if result.returncode != 0:
    print(result.stderr)
    raise RuntimeError("❌ Unzip failed. File may be corrupted or unsupported.")
else:
    print("✅ Data extracted successfully using system unzip!")

Forcefully clearing and recreating mount point directory: /content/drive
Error removing /content/drive: [Errno 125] Operation canceled: '/content/drive/.Encrypted/MyDrive'. This might happen if it's currently in use or due to permissions.
Google Drive unmounted successfully before remount attempt.
Attempting to mount Google Drive...
Mounted at /content/drive
✅ Zip file found: /content/drive/MyDrive/Nat2024us.zip
Unzipping with system tool...
✅ Data extracted successfully using system unzip!


In [ ]:
# =========================
# 5) FIND THE DATA FILE
# =========================
# CDC files are usually .txt or .dat
candidate_files = []

for root, dirs, files in os.walk(EXTRACT_DIR):
    for file in files:
        lower = file.lower()
        if lower.endswith(".txt") or lower.endswith(".dat"):
            full_path = os.path.join(root, file)
            candidate_files.append(full_path)

if not candidate_files:
    raise FileNotFoundError(
        "No .txt or .dat file found after unzipping. Check what is inside the zip."
    )

# Usually the largest .txt/.dat file is the natality data file
DATA_FILE = max(candidate_files, key=lambda p: os.path.getsize(p))

print("\n✅ Found candidate data files:")
for f in candidate_files:
    print("-", f)

print("\n✅ Using data file:", DATA_FILE)

# =========================
# 6) READ SELECTED FIXED-WIDTH COLUMNS
# =========================
# Positions are from the 2024 User Guide.
# Python colspecs are 0-based, end-exclusive.

colspecs = [
    (8, 12),    # DOB_YY      9-12
    (74, 76),   # MAGER       75-76
    (123, 124), # MEDUC       124
    (250, 251), # WIC         251
    (251, 252), # F_WIC       252
    (286, 287), # BMI_R       287
    (312, 313), # RF_PDIAB    313
    (313, 314), # RF_GDIAB    314
    (314, 315), # RF_PHYPE    315
    (315, 316), # RF_GHYPE    316
    (316, 317), # RF_EHYPE    317
    (317, 318), # RF_PPTERM   318
    (414, 415), # MM_MTR      415
    (415, 416), # MM_PLAC     416
    (416, 417), # MM_RUPT     417
    (417, 418), # MM_UHYST    418
    (418, 419), # MM_AICU     419
    (426, 427), # NO_MMORB    427
    (434, 435), # PAY         435
    (453, 454), # DPLURAL     454
]

names = [
    "DOB_YY",
    "MAGER",
    "MEDUC",
    "WIC",
    "F_WIC",
    "BMI_R",
    "RF_PDIAB",
    "RF_GDIAB",
    "RF_PHYPE",
    "RF_GHYPE",
    "RF_EHYPE",
    "RF_PPTERM",
    "MM_MTR",
    "MM_PLAC",
    "MM_RUPT",
    "MM_UHYST",
    "MM_AICU",
    "NO_MMORB",
    "PAY",
    "DPLURAL",
]

print("\nReading fixed-width file...")
df = pd.read_fwf(
    DATA_FILE,
    colspecs=colspecs,
    names=names,
    dtype=str
)

# Keep exactly 10% of the data to stay under 25 MB
df = df.sample(frac=0.10, random_state=42)

print("✅ Raw shape:", df.shape)
print(df.head())

# =========================
# 7) BASIC CLEANING
# =========================
for col in df.columns:
    df[col] = df[col].astype(str).str.strip()

# Standardize blanks
df = df.replace({
    "": np.nan,
    "nan": np.nan,
    "None": np.nan
})

# Keep only 2024 rows
df = df[df["DOB_YY"] == "2024"].copy()

print("\n✅ Shape after year filter:", df.shape)


✅ Found candidate data files:
- /content/nat2024_project/extracted/Nat2024PublicUS.c20250512.r20250708.txt

✅ Using data file: /content/nat2024_project/extracted/Nat2024PublicUS.c20250512.r20250708.txt

Reading fixed-width file...
✅ Raw shape: (363844, 20)
        DOB_YY MAGER MEDUC WIC F_WIC BMI_R RF_PDIAB RF_GDIAB RF_PHYPE  \
2839139   2024    21     3   Y     1     2        N        N        N   
2650778   2024    36     7   N     1     2        N        N        N   
2172738   2024    34     5   N     1     5        N        N        N   
2792799   2024    36     6   N     1     2        N        N        N   
1699796   2024    27     6   N     1     3        N        N        Y   

        RF_GHYPE RF_EHYPE RF_PPTERM MM_MTR MM_PLAC MM_RUPT MM_UHYST MM_AICU  \
2839139        N        N         Y      N       N       N        N       N   
2650778        Y        N         N      N       N       N        N       N   
2172738        N        N         Y      N       N       N        

In [ ]:
# =========================
# 8) FEATURE ENGINEERING
# =========================

# ----- numeric versions -----
df["year"] = pd.to_numeric(df["DOB_YY"], errors="coerce")
df["maternal_age"] = pd.to_numeric(df["MAGER"], errors="coerce")

# ----- education -----
educ_map = {
    "1": "8th grade or less",
    "2": "9th-12th no diploma",
    "3": "High school/GED",
    "4": "Some college no degree",
    "5": "Associate degree",
    "6": "Bachelor degree",
    "7": "Master degree",
    "8": "Doctorate/professional",
    "9": "Unknown"
}
df["education"] = df["MEDUC"].map(educ_map)

# ----- insurance -----
pay_map = {
    "1": "Medicaid",
    "2": "Private insurance",
    "3": "Self-pay",
    "4": "Indian Health Service",
    "5": "CHAMPUS/TRICARE",
    "6": "Other government",
    "8": "Other",
    "9": "Unknown"
}
df["insurance_status"] = df["PAY"].map(pay_map)

# ----- WIC as income proxy -----
wic_map = {
    "Y": 1,
    "N": 0,
    "U": np.nan
}
df["wic_used"] = df["WIC"].map(wic_map)

f_wic_map = {
    "0": 0,
    "1": 1
}
df["wic_reporting_flag"] = df["F_WIC"].map(f_wic_map)

# ----- obesity -----
# BMI_R codes 4,5,6 are obesity categories
df["obesity"] = df["BMI_R"].isin(["4", "5", "6"]).astype(int)

bmi_cat_map = {
    "1": "Underweight",
    "2": "Normal",
    "3": "Overweight",
    "4": "Obesity I",
    "5": "Obesity II",
    "6": "Extreme Obesity III",
    "9": "Unknown"
}
df["bmi_category"] = df["BMI_R"].map(bmi_cat_map)

# ----- plurality / multiple gestations -----
plural_map = {
    "1": "Single",
    "2": "Twin",
    "3": "Triplet",
    "4": "Quadruplet or higher"
}
df["plurality_category"] = df["DPLURAL"].map(plural_map)
df["multiple_gestations"] = df["DPLURAL"].isin(["2", "3", "4"]).astype(int)

# ----- yes/no/unknown mapping -----
ynu_map = {
    "Y": 1,
    "N": 0,
    "U": np.nan,
    "X": np.nan
}

risk_cols = [
    "RF_PDIAB",
    "RF_GDIAB",
    "RF_PHYPE",
    "RF_GHYPE",
    "RF_EHYPE",
    "RF_PPTERM"
]

for col in risk_cols:
    df[col + "_bin"] = df[col].map(ynu_map)

# Hypertensive disease from the three hypertension-related fields
df["hypertensive_disease"] = (
    df[["RF_PHYPE_bin", "RF_GHYPE_bin", "RF_EHYPE_bin"]]
    .max(axis=1, skipna=True)
)

# Comorbidity count across extracted risk-factor fields
df["comorbidity_count"] = (
    df[[col + "_bin" for col in risk_cols]]
    .fillna(0)
    .sum(axis=1)
    .astype(int)
)

# =========================
# 9) CREATE TARGET VARIABLE: SMM
# =========================
# SMM = 1 if any maternal morbidity field is Y
morbidity_cols = ["MM_MTR", "MM_PLAC", "MM_RUPT", "MM_UHYST", "MM_AICU"]

for col in morbidity_cols:
    df[col + "_bin"] = df[col].map(ynu_map)

df["SMM"] = (
    df[[col + "_bin" for col in morbidity_cols]]
    .fillna(0)
    .max(axis=1)
    .astype(int)
)

# =========================
# 10) FINAL DATASET
# =========================
final_cols = [
    "year",
    "maternal_age",
    "insurance_status",
    "education",
    "wic_used",
    "obesity",
    "multiple_gestations",
    "hypertensive_disease",
    "comorbidity_count",
    "SMM"
]

final_df = df[final_cols].copy()

# Remove clearly impossible maternal ages if present
final_df = final_df[
    final_df["maternal_age"].isna() |
    ((final_df["maternal_age"] >= 10) & (final_df["maternal_age"] <= 60))
].copy()

print("\n✅ Final dataset shape:", final_df.shape)
print("\n✅ Target balance:")
print(final_df["SMM"].value_counts(dropna=False))
print(final_df["SMM"].value_counts(normalize=True, dropna=False))

print("\n✅ Preview:")
print(final_df.head())

# =========================
# 11) SAVE CLEANED DATASET
# =========================
clean_csv = OUTPUT_DIR / "DATASET.csv"
sample_xlsx = OUTPUT_DIR / "DATASET.xlsx"

# Save full dataset as CSV
final_df.to_csv(clean_csv, index=False)

# Save a smaller Excel sample only
final_df.sample(n=min(50000, len(final_df)), random_state=42).to_excel(sample_xlsx, index=False)

print("\n✅ Saved cleaned files:")
print(clean_csv)
print(sample_xlsx)


✅ Final dataset shape: (363844, 10)

✅ Target balance:
SMM
0    358445
1      5399
Name: count, dtype: int64
SMM
0    0.985161
1    0.014839
Name: proportion, dtype: float64

✅ Preview:
         year  maternal_age   insurance_status         education  wic_used  \
2839139  2024            21           Medicaid   High school/GED       1.0   
2650778  2024            36  Private insurance     Master degree       0.0   
2172738  2024            34  Private insurance  Associate degree       0.0   
2792799  2024            36  Private insurance   Bachelor degree       0.0   
1699796  2024            27           Medicaid   Bachelor degree       0.0   

         obesity  multiple_gestations  hypertensive_disease  \
2839139        0                    0                   0.0   
2650778        0                    0                   1.0   
2172738        1                    0                   0.0   
2792799        0                    0                   0.0   
1699796        0             

In [ ]:
# =========================
# 12) TRAIN / VALIDATION / TEST SPLIT
# =========================
X = final_df.drop(columns=["SMM"])
y = final_df["SMM"].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

X_train, X_val, y_train, y_val = train_test_split(
    X_train,
    y_train,
    test_size=0.25,   # 0.25 of 0.80 = 0.20 total validation
    random_state=42,
    stratify=y_train
)

train_df = X_train.copy()
train_df["SMM"] = y_train.values

val_df = X_val.copy()
val_df["SMM"] = y_val.values

test_df = X_test.copy()
test_df["SMM"] = y_test.values

train_csv = OUTPUT_DIR / "train_dataset_2024.csv"
val_csv = OUTPUT_DIR / "validation_dataset_2024.csv"
test_csv = OUTPUT_DIR / "test_dataset_2024.csv"

train_df.to_csv(train_csv, index=False)
val_df.to_csv(val_csv, index=False)
test_df.to_csv(test_csv, index=False)

print("\n✅ Saved split datasets:")
print(train_csv)
print(val_csv)
print(test_csv)


✅ Saved split datasets:
/content/nat2024_project/outputs/train_dataset_2024.csv
/content/nat2024_project/outputs/validation_dataset_2024.csv
/content/nat2024_project/outputs/test_dataset_2024.csv


In [ ]:
# =========================
# 13) PREPROCESSING
# =========================
numeric_features = [
    "year",
    "maternal_age",
    "wic_used",
    "obesity",
    "multiple_gestations",
    "hypertensive_disease",
    "comorbidity_count"
]

categorical_features = [
    "insurance_status",
    "education"
]

from sklearn.preprocessing import StandardScaler # Add this import at the top of your script

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()) # THIS IS CRITICAL FOR LASSO/SAGA
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

In [ ]:
# =========================
# 14) MODELS WITH CLASS IMBALANCE HANDLING
# =========================
models = {
    "LogisticRegression_balanced": LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        random_state=42
    ),

    "Lasso_Classifier": LogisticRegression(
        penalty="l1",
        solver="liblinear",
        max_iter=2000,
        class_weight="balanced",
        random_state=42
    )
}

results = []

for model_name, model in models.items():
    clf = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("model", model)
    ])

    clf.fit(X_train, y_train)

    val_pred = clf.predict(X_val)

    if hasattr(clf.named_steps["model"], "predict_proba"):
        val_scores = clf.predict_proba(X_val)[:, 1]
    else:
        val_scores = clf.decision_function(X_val)

    f1 = f1_score(y_val, val_pred, zero_division=0)
    recall = recall_score(y_val, val_pred, zero_division=0)
    precision = precision_score(y_val, val_pred, zero_division=0)
    auprc = average_precision_score(y_val, val_scores)

    results.append({
        "model": model_name,
        "f1": f1,
        "recall": recall,
        "precision": precision,
        "auprc": auprc,
        "pipeline": clf
    })

results_df = pd.DataFrame(results).drop(columns=["pipeline"]).sort_values(
    by=["auprc", "f1", "recall"],
    ascending=False
)

print("\n✅ Validation results:")
print(results_df)

best_model_name = results_df.iloc[0]["model"]
best_pipeline = next(r["pipeline"] for r in results if r["model"] == best_model_name)


✅ Validation results:
                         model        f1    recall  precision     auprc
1             Lasso_Classifier  0.038869  0.567593   0.020123  0.022017
0  LogisticRegression_balanced  0.038737  0.566667   0.020054  0.021999


In [ ]:
# =========================
# 15) FINAL TEST EVALUATION
# =========================
test_pred = best_pipeline.predict(X_test)

if hasattr(best_pipeline.named_steps["model"], "predict_proba"):
    test_scores = best_pipeline.predict_proba(X_test)[:, 1]
else:
    test_scores = best_pipeline.decision_function(X_test)

test_f1 = f1_score(y_test, test_pred, zero_division=0)
test_recall = recall_score(y_test, test_pred, zero_division=0)
test_precision = precision_score(y_test, test_pred, zero_division=0)
test_auprc = average_precision_score(y_test, test_scores)
test_cm = confusion_matrix(y_test, test_pred)
test_report = classification_report(y_test, test_pred, zero_division=0)

print("\n✅ Best model:", best_model_name)
print("Test F1:", test_f1)
print("Test Recall:", test_recall)
print("Test Precision:", test_precision)
print("Test AUPRC:", test_auprc)
print("\nConfusion Matrix:\n", test_cm)
print("\nClassification Report:\n", test_report)

# =========================
# 16) SAVE METRICS
# =========================
metrics_file = OUTPUT_DIR / "model_metrics_2024.txt"

with open(metrics_file, "w", encoding="utf-8") as f:
    f.write("CDC 2024 Natality Modeling Results\n")
    f.write("=" * 50 + "\n\n")
    f.write("Target definition:\n")
    f.write("SMM = 1 if any of MM_MTR, MM_PLAC, MM_RUPT, MM_UHYST, MM_AICU == Y\n\n")

    f.write("Validation results:\n")
    f.write(results_df.to_string(index=False))
    f.write("\n\n")

    f.write(f"Best model: {best_model_name}\n\n")
    f.write(f"Test F1: {test_f1:.6f}\n")
    f.write(f"Test Recall: {test_recall:.6f}\n")
    f.write(f"Test Precision: {test_precision:.6f}\n")
    f.write(f"Test AUPRC: {test_auprc:.6f}\n\n")

    f.write("Confusion Matrix:\n")
    f.write(str(test_cm))
    f.write("\n\n")

    f.write("Classification Report:\n")
    f.write(test_report)

print("\n✅ Saved metrics file:")
print(metrics_file)

print("\n🎉 DONE — All outputs are in:")
print(OUTPUT_DIR)


✅ Best model: LogisticRegression_balanced
Test F1: 0.03558191252779837
Test Recall: 0.536741214057508
Test Precision: 0.018400876232201532
Test AUPRC: 0.01899426554378112

Confusion Matrix:
 [[10725  8962]
 [  145   168]]

Classification Report:
               precision    recall  f1-score   support

           0       0.99      0.54      0.70     19687
           1       0.02      0.54      0.04       313

    accuracy                           0.54     20000
   macro avg       0.50      0.54      0.37     20000
weighted avg       0.97      0.54      0.69     20000


✅ Saved metrics file:
/content/nat2024_project/outputs/model_metrics_2024.txt

🎉 DONE — All outputs are in:
/content/nat2024_project/outputs


In [ ]:
import shutil

# Note: Capitalization matters in Colab. Usually it is 'MyDrive', not 'Mydrive'
drive_output = "/content/drive/MyDrive/Final_outputs"
os.makedirs(drive_output, exist_ok=True)

# Change the source to the actual local output folder you defined at the top of the script!
shutil.copytree(OUTPUT_DIR, drive_output, dirs_exist_ok=True)

print("Files copied to google drive", drive_output)

Files copied to google drive /content/drive/MyDrive/Final_outputs
